In [1]:
import json
import os
import requests
import time
from dotenv import load_dotenv
from pathlib import Path
import yfinance as yf
import pandas as pd

from util import *


load_dotenv(override=True)

ALPHA_VANTAGE_KEY = os.getenv('ALPHA_VANTAGE_KEY')
IDENTITY = os.getenv("IDENTITY")
CHECKPOINT_FILE = Path("checkpoint.json")

BASE_URL = "https://www.alphavantage.co/query"

COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_HEADERS = {
    "User-Agent": f"financial-research-masters-degree ({IDENTITY})",
    "Accept": "application/json"
}

FUNCTIONS = {
    "INCOME_STATEMENT": "income_statement.json",
    "BALANCE_SHEET": "balance_sheet.json",
    "CASH_FLOW": "cash_flow.json"
}

RAW_DIR = Path("raw_data")
RAW_DIR.mkdir(exist_ok=True)

In [2]:

def fetch(function: str, ticker: str):
    params = {
        "function": function,
        "symbol": ticker,
        "apikey": ALPHA_VANTAGE_KEY
    }

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()

    data = r.json()

    if "Information" in data or "Note" in data:
        print(f"Error: {data.get("Information")}")
        raise RuntimeError("ALPHA_VANTAGE_LIMIT")
    
    if "quarterlyReports" not in data and "annualReports" not in data:
        raise RuntimeError("INVALID_ALPHA_RESPONSE")

    return data

def download_company(ticker: str):
    company_dir = RAW_DIR / ticker
    company_dir.mkdir(exist_ok=True)

    for func, filename in FUNCTIONS.items():

        filepath = company_dir / filename

        if filepath.exists():
            print("skip", ticker, filename)
            continue

        print("download", ticker, func)

        data = fetch(func, ticker)

        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)

        # Alpha Vantage rate limit
        time.sleep(12)



In [3]:

def load_raw(ticker: str):
    base = RAW_DIR / ticker

    with open(base / "income_statement.json") as f:
        income = json.load(f)

    with open(base / "balance_sheet.json") as f:
        balance = json.load(f)

    with open(base / "cash_flow.json") as f:
        cash = json.load(f)

    return income, balance, cash

def merge_reports(income: dict, balance: dict, cash: dict, key: str):
    '''
        key can be quarterlyReports | annualReports
    '''

    merged = {}

    for row in income.get(key, []):
        d = row["fiscalDateEnding"]
        merged.setdefault(d, {}).update(row)

    for row in balance.get(key, []):
        d = row["fiscalDateEnding"]
        merged.setdefault(d, {}).update(row)

    for row in cash.get(key, []):
        d = row["fiscalDateEnding"]
        merged.setdefault(d, {}).update(row)

    result = [
        {"fiscalDateEnding": k, **{ik: iv for ik, iv in v.items() if ik != "fiscalDateEnding"}}
        for k, v in merged.items()
        if k >= "2010-01-01"
    ]

    result.sort(key=lambda x: x["fiscalDateEnding"], reverse=False)
    return result


def build_reports(ticker: str, metadata: dict, period: str):
    """
    period: 'quarterly' | 'yearly'
    """
    assert period in ('quarterly', 'yearly'), "period must be 'quarterly' or 'yearly'"

    report_key = 'quarterlyReports' if period == 'quarterly' else 'annualReports'
    output_dir = f"data_{period}"

    income, balance, cash = load_raw(ticker)
    merged = merge_reports(income, balance, cash, report_key)

    price_history = get_price_history(ticker)
    prev_data = None
    for i, frame_data in enumerate(merged):
        fiscalDateEnding = frame_data.get("fiscalDateEnding")
        frame_data['stock_price'] = lookup_price(price_history, fiscalDateEnding)

        frame_data = sanitize_metrics(frame_data)
        apply_accounting_logic_af(frame_data)

        indicators = compute_indicators(frame_data, prev_data, period)
        prev_data = frame_data
        for k, v in indicators.items():
            frame_data[f"ind_{k}"] = v

        merged[i] = frame_data

    sector_name = metadata["sector"].lower().replace(" ", "_")
    Path(f"{output_dir}/{sector_name}").mkdir(exist_ok=True)

    df = pd.DataFrame(merged)
    df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])
    df.sort_values('fiscalDateEnding', ascending=False, inplace=True)
    df.to_csv(f"{output_dir}/{sector_name}/{ticker}.csv", index=False)

    print(f"Saved data to {output_dir}/{sector_name}/{ticker}.csv")

    save_metadata(ticker, metadata, f"{output_dir}/metadata.json")

def process_ticker(ticker: str, metadata: dict):
    build_reports(ticker,  metadata, 'quarterly')
    build_reports(ticker,  metadata, 'yearly')

def load_checkpoint_data() -> int:
    if not CHECKPOINT_FILE.exists():
        return 0, []
    
    with open(CHECKPOINT_FILE, 'r') as f:
        data = json.load(f)

    return data["last_completed_index"] + 1, data["skip_tickers"]

def save_checkpoint_data(index: int, tickers_to_skip: list):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({"last_completed_index": index, "skip_tickers": tickers_to_skip}, f)

def is_company_downloaded(ticker):

    base = Path("raw_data") / ticker

    return (
        (base / "income_statement.json").exists() and
        (base / "balance_sheet.json").exists() and
        (base / "cash_flow.json").exists()
    )


def get_company_tickers():
    if os.path.exists('company_tickers.json'):
        with open('company_tickers.json', 'r') as f:
            data = json.load(f)
    else:
        r = requests.get(COMPANY_TICKERS_URL, headers=SEC_HEADERS)
        r.raise_for_status()
        data = r.json()
    return data


In [39]:
tickers = get_company_tickers()

start_index, tickers_to_skip = load_checkpoint_data()

for i in range(start_index, len(tickers)-1):
    info = tickers[str(i)]
    ticker = info.get('ticker')
    title = info.get('title')
    # ticker = "AAPL"
    # title = ""
    if ticker in tickers_to_skip:
        print(f"Ticker: {ticker} previously returned invalid data, skipping")
        continue

    # Maybe only download data from sectors
    # ('Technology', 'Financial Services', 'Healthcare', 'Industrials')
    # to save requests daily?
    metadata = get_company_metadata(ticker)
    if not metadata or metadata["sector"] not in ('Technology', 'Financial Services', 'Healthcare', 'Industrials'):
        print(f'Skipping {title} from sector: {metadata.get("sector", "unknown")}')
        continue


    print(f"Downloading Alpha Vantage data for {ticker}:")

    try:
        download_company(ticker)
        
        if is_company_downloaded(ticker):
            process_ticker(ticker, metadata)
            save_checkpoint_data(i, tickers_to_skip)
    except RuntimeError as e:
        if str(e) == "ALPHA_VANTAGE_LIMIT":
            print("Alpha Vantage limit reached. Stopping.")
            break
        else:
            print("Error:", ticker, e)
            tickers_to_skip.append(ticker)
            save_checkpoint_data(i, tickers_to_skip)
            os.rmdir(os.path.join("raw_data", ticker))
            time.sleep(1)

Skipping NIKE, Inc. from sector: Consumer Cyclical
skip CDNS income_statement.json
download CDNS BALANCE_SHEET
Error: We have detected your API key as APWF7XFG2JF6ZCGE and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.
Alpha Vantage limit reached. Stopping.


In [4]:
for ticker in os.listdir("raw_data"):
    if len(os.listdir(f"raw_data/{ticker}")) < 3:
        print(f"Skipping {ticker}, invalid")
        continue
    metadata = get_company_metadata(ticker)
    # if not os.path.exists(f'data_quarterly/{metadata['sector'].lower().replace(" ", "_")}/{ticker}.csv') or not os.path.exists(f'data_yearly/{metadata['sector'].lower().replace(" ", "_")}/{ticker}.csv'):
    process_ticker(ticker, metadata)
    # else:
    #     print(f"{ticker} already parsed")
    break


Saved data to data_quarterly/financial_services/IBKR.csv
Saved data to data_yearly/financial_services/IBKR.csv


In [ ]:
with open(f'data_yearly/metadata.json', 'r') as f:
    metadata = json.load(f)

{'sector': 'Technology', 'industry': 'Consumer Electronics'}

In [ ]:
def export_data_for_training(
        data_dir: str, 
        sectors: tuple = ('technology', 'financial_services', 'healthcare', 'industrials')
    ) -> pd.DataFrame:
    with open(f'{data_dir}/metadata.json', 'r') as f:
        metadata = json.load(f)

    # for sector in os.listdir(data_dir):
    frames = []
    for sector in sectors:
        for file in os.listdir(os.path.join(data_dir, sector)):
            if file.endswith('.csv'):
                base = pd.read_csv(os.path.join(data_dir, sector, file))
                extra = pd.DataFrame({
                    "ticker": file[:-4],
                    "sector": metadata[file[:-4]].get("sector"),
                }, index=base.index)

                df = pd.concat([base, extra], axis=1)
                frames.append(df)

    if frames:
        return pd.concat(frames, ignore_index=True)
    else:
        print("Warnign: No frames found, returning empty DF")
        return pd.DataFrame()


    

In [ ]:
frames = []

DIR = "data_quarterly"
total_missing = 0
total = 0

for sector in ('financial_services', 'healthcare', 'industrials', 'technology'):
# for sector in os.listdir(DIR):
#     if sector.endswith('.json'):
#         continue
    for file in os.listdir(os.path.join(DIR, sector)):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(DIR, sector, file))
            df = df.filter(like="ind_")
            print(f"{file} missing: {df.isna().sum().sum()} / {len(df) * len(df.columns)} indicators")
            total_missing += df.isna().sum().sum()
            total += len(df) * len(df.columns)
            frames.append(df)

if frames:
    final_df = pd.concat(frames, ignore_index=True)

print("Final statistics:")
print(f"total missing ratio: {total_missing} / {total} => {round(total_missing / total * 100, 2)}%")
print(f'Data files: {len(frames)}')
(final_df.filter(like="ind_").isna().mean() * 100).round(2)

MUFG.csv missing: 1110 / 3526 indicators
IBKR.csv missing: 147 / 3483 indicators
WFC.csv missing: 41 / 3483 indicators
BLK.csv missing: 16 / 3483 indicators
BRK-B.csv missing: 410 / 3483 indicators
AXP.csv missing: 22 / 3483 indicators
CME.csv missing: 42 / 3483 indicators
BAC.csv missing: 32 / 3483 indicators
COF.csv missing: 19 / 3483 indicators
GS.csv missing: 13 / 3483 indicators
MA.csv missing: 151 / 3483 indicators
RY.csv missing: 31 / 3483 indicators
MS.csv missing: 48 / 3483 indicators
HDB.csv missing: 467 / 3483 indicators
TD.csv missing: 22 / 3483 indicators
C.csv missing: 192 / 3483 indicators
CB.csv missing: 106 / 3483 indicators
BMO.csv missing: 69 / 3483 indicators
V.csv missing: 280 / 3397 indicators
HSBC.csv missing: 1494 / 3870 indicators
SCHW.csv missing: 12 / 3483 indicators
SAN.csv missing: 596 / 3483 indicators
JPM.csv missing: 19 / 3483 indicators
NFLX.csv missing: 171 / 3483 indicators
DIS.csv missing: 47 / 3483 indicators
T.csv missing: 37 / 3483 indicators
VZ.c

ind_grossMargin                       1.00
ind_operatingMargin                   0.99
ind_netProfitMargin                   1.01
ind_ebitdaMargin                      1.72
ind_fcfMargin                         3.39
ind_roe                               1.70
ind_roa                               1.48
ind_equityMultiplier                  1.83
ind_roic                              6.72
ind_investedCapital                   6.51
ind_cashKingRatio                    10.66
ind_ocfToRevenue                      3.39
ind_fcfAfterSbcDilutionCost          20.04
ind_fcfAfterSbcDilutionCostMargin    20.04
ind_sbcToRevenue                     19.69
ind_ownerEarnings                     2.66
ind_sloanRatio                        5.97
ind_currentRatio                      2.68
ind_quickRatio                        2.68
ind_debtToEquity                      6.51
ind_debtToAssets                      6.40
ind_interestCoverage                  4.96
ind_netDebtToEbitda                   6.83
ind_eps    

In [8]:
import yfinance as yf

# Count SPY value as S&P (market) value for comparison:
ticker = yf.Ticker('SPY')
hist = ticker.history(period='max')
hist.index = hist.index.tz_localize(None)

lookup_price(hist, '2026-4-2')


np.float64(655.83)